In [4]:
import os
import re
import cv2
import numpy as np
import pandas as pd
from tkinter import Tk, filedialog
import tensorflow as tf
from ultralytics import YOLO

# ========================= CONFIG =========================
NUM_DIGITS = 5           
DIGIT_HEIGHT = 32         
DIGIT_WIDTH = 32

MIN_COMPONENT_AREA_FRAC = 0.01   
MAX_COMPONENT_AREA_FRAC = 0.60   
BOX_PADDING_PX = 2               

YOLO_MODEL_PATH = r"best.pt"
CNN_MODEL_PATH = "custom.keras"

# ========================= LOAD MODELS =========================
if not os.path.exists(YOLO_MODEL_PATH):
    raise FileNotFoundError(f"Can't find YOLO weights at {YOLO_MODEL_PATH}")
if not os.path.exists(CNN_MODEL_PATH):
    raise FileNotFoundError(f"Can't find CNN model at {CNN_MODEL_PATH}")

yolo_model = YOLO(YOLO_MODEL_PATH)
digit_model = tf.keras.models.load_model(CNN_MODEL_PATH, compile=False)
print(f"Loaded YOLO from {YOLO_MODEL_PATH} and CNN from {CNN_MODEL_PATH}")


# ========================= PIPELINE FUNCTIONS =========================

def get_meter_roi(image_path, yolo_model=yolo_model, conf=0.25):
    """Runs YOLO on an image path, returns highest confidence ROI as BGR array."""
    # Read image explicitly to handle paths safely
    img = cv2.imread(image_path)
    if img is None:
        return None
        
    result = yolo_model.predict(source=img, imgsz=640, conf=conf, verbose=False)[0]
    if len(result.boxes) == 0:
        return None

    boxes = result.boxes.xyxy.cpu().numpy().astype(int)
    confidences = result.boxes.conf.cpu().numpy()
    x1, y1, x2, y2 = boxes[int(np.argmax(confidences))]

    roi = result.orig_img[y1:y2, x1:x2]
    return roi if roi.size > 0 else None


def _binarize_for_segmentation(gray_img):
    denoised = cv2.bilateralFilter(gray_img, d=5, sigmaColor=50, sigmaSpace=50)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(binary == 255) > 0.5:
        binary = cv2.bitwise_not(binary)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    return binary


def _split_merged_box(binary_crop, expected_splits):
    col_sums = binary_crop.sum(axis=0)
    w = binary_crop.shape[1]
    seg_width = max(1, w // expected_splits)

    split_points = [0]
    for i in range(1, expected_splits):
        approx = i * seg_width
        lo = max(0, approx - seg_width // 3)
        hi = min(w, approx + seg_width // 3)
        window = col_sums[lo:hi]
        if len(window) == 0:
            split_points.append(approx)
            continue
        valley_offset = int(np.argmin(window))
        split_points.append(lo + valley_offset)
    split_points.append(w)

    split_points = sorted(set(split_points))
    if len(split_points) < expected_splits + 1:
        split_points = [int(i * w / expected_splits) for i in range(expected_splits + 1)]

    return [(split_points[i], split_points[i + 1]) for i in range(len(split_points) - 1)]


def _split_oversized_boxes(boxes, binary_img, num_digits):
    if len(boxes) == 0:
        return boxes
    widths = [b[2] for b in boxes]
    median_w = np.median(widths)

    new_boxes = []
    for (x, y, w, h) in boxes:
        if median_w > 0 and w > median_w * 1.6:
            n_merged = max(2, round(w / median_w))
            n_merged = min(n_merged, num_digits)  
            sub_crop = binary_img[y:y + h, x:x + w]
            for (sx0, sx1) in _split_merged_box(sub_crop, n_merged):
                new_boxes.append((x + sx0, y, sx1 - sx0, h))
        else:
            new_boxes.append((x, y, w, h))

    new_boxes.sort(key=lambda b: b[0])
    return new_boxes


def segment_digit_boxes(gray_img, num_digits=NUM_DIGITS):
    h_img, w_img = gray_img.shape[:2]
    img_area = h_img * w_img

    binary = _binarize_for_segmentation(gray_img)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

    boxes = []
    for label_id in range(1, num_labels):  
        x, y, w, h, area = stats[label_id]
        area_frac = area / img_area
        if MIN_COMPONENT_AREA_FRAC <= area_frac <= MAX_COMPONENT_AREA_FRAC:
            boxes.append((x, y, w, h))

    boxes.sort(key=lambda b: b[0])
    if len(boxes) > 0:
        boxes = _split_oversized_boxes(boxes, binary, num_digits)

    if len(boxes) == num_digits:
        return boxes

    if len(boxes) > num_digits:
        boxes.sort(key=lambda b: b[2] * b[3], reverse=True)
        boxes = boxes[:num_digits]
        boxes.sort(key=lambda b: b[0])
        return boxes

    col_w = w_img // num_digits
    return [(i * col_w, 0, col_w, h_img) for i in range(num_digits)]


def extract_digit_crops(gray_img, num_digits=NUM_DIGITS, target_size=(DIGIT_HEIGHT, DIGIT_WIDTH)):
    h_img, w_img = gray_img.shape[:2]
    boxes = segment_digit_boxes(gray_img, num_digits=num_digits)

    crops = []
    for (x, y, w, h) in boxes:
        x0 = max(0, x - BOX_PADDING_PX)
        y0 = max(0, y - BOX_PADDING_PX)
        x1 = min(w_img, x + w + BOX_PADDING_PX)
        y1 = min(h_img, y + h + BOX_PADDING_PX)

        crop = gray_img[y0:y1, x0:x1]
        if crop.size == 0:
            crop = np.zeros((target_size[0], target_size[1]), dtype=np.uint8)

        crop = cv2.resize(crop, (target_size[1], target_size[0]))
        crop = crop.astype(np.float32) / 255.0
        crops.append(np.expand_dims(crop, axis=-1))

    return crops


def predict_meter_reading(roi_bgr, num_digits=NUM_DIGITS, model=digit_model):
    """Processes ROI and returns the string prediction without plotting windows."""
    gray_img = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
    crops = extract_digit_crops(gray_img, num_digits=num_digits)

    batch = np.stack(crops, axis=0)  
    preds = model.predict(batch, verbose=0)
    digit_predictions = np.argmax(preds, axis=1)
    
    return "".join(map(str, digit_predictions))


# ========================= UTILITY FUNCTIONS =========================

def select_folder():
    """Open a dialog to pick the folder containing the images."""
    root = Tk()
    root.withdraw()          
    root.attributes("-topmost", True)
    folder_path = filedialog.askdirectory(title="Select Folder with Meter Images")
    root.destroy()
    return folder_path


def parse_actual_label(filename, num_digits=NUM_DIGITS):
    """Tries to find a sequence of exactly `num_digits` numerical characters 
    inside the filename to use as the ground truth."""
    match = re.search(r'\d{' + str(num_digits) + r'}', filename)
    return match.group(0) if match else "Unknown"


# ========================= BATCH PROCESSING MAIN =========================

if __name__ == "__main__":
    folder_path = select_folder()

    if not folder_path:
        print("No folder selected.")
    else:
        # Supported extensions
        valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
        image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]

        if not image_files:
            print(f"No valid images found in {folder_path}")
        else:
            results = []
            print(f"Processing {len(image_files)} images from: {folder_path}\n")

            for filename in image_files:
                full_path = os.path.join(folder_path, filename)
                actual_label = parse_actual_label(filename, NUM_DIGITS)
                
                try:
                    roi = get_meter_roi(full_path, yolo_model=yolo_model, conf=0.25)
                    
                    if roi is None:
                        predicted_label = "No Meter Detected"
                    else:
                        predicted_label = predict_meter_reading(roi, num_digits=NUM_DIGITS, model=digit_model)
                        
                except Exception as e:
                    predicted_label = f"Error ({str(e)})"

                results.append({
                    "Image Name": filename,
                    "Actual Label": actual_label,
                    "Predicted Label": predicted_label
                })

            # Create Pandas DataFrame and display as a beautiful table
            df = pd.DataFrame(results)
            
            print("\n========================= METRIC RESULTS =========================")
            # Adjust pandas viewing configurations so text doesn't get clipped in terminal
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', 1000)
            print(df.to_string(index=False))
            print("===================================================================\n")
            
            # Optional: Save results to CSV in the source folder
            output_csv = os.path.join(folder_path, "batch_predictions.csv")
            df.to_csv(output_csv, index=False)
            print(f"Results successfully saved to: {output_csv}")

Loaded YOLO from best.pt and CNN from custom.keras
Processing 150 images from: D:/internship/internship_computer_visions_engineering/data_Set/validation/images


========================= METRIC RESULTS =========================
                    Image Name Actual Label   Predicted Label
          0642_73167377834.jpg        73167             00023
          0642_73608310006.jpg        73608             00090
          0642_73697310008.jpg        73697             17370
          0642_74369257212.jpg        74369             00015
          0642_74497310008.jpg        74497             00026
          0642_75668028337.jpg        75668             90925
          0642_75697310003.jpg        75697             00088
          0642_76274002658.jpg        76274             00011
          0642_77121344194.jpg        77121             00018
          0642_77397310002.jpg        77397             00009
          0642_78568604579.jpg        78568             00008
          0642_78668786359.